# 03 - End-to-End Pipeline

This notebook runs the full Colab pipeline for encoder integration, feature extraction, CLIP target generation, and MLP training:
- environment setup
- Concerto package installation
- encoder execution on a real Area 5 room
- feature extraction with `scripts/extract_features.py`
- CLIP label embedding generation if needed
- MLP training with `src/train.py`
- checkpoint inspection


### 1. Mount Drive and Get the Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_DIR = '/content/Deep_learning_project'
BRANCH = 'main'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Gandata/Deep_learning_project.git {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin
    !git pull origin {BRANCH}

%cd {REPO_DIR}


### 2. Install Project Dependencies

In [ ]:
!pip install -q -r requirements.txt


### 3. Install Concerto Package Mode

This follows the official Concerto package-mode setup: install `spconv`, `torch-scatter`, `timm`, then install the Concerto repo itself.

In [ ]:
import os
import torch

if torch.version.cuda is None:
    raise RuntimeError(
        'Colab is running without CUDA. Switch to Runtime > Change runtime type > T4 GPU, then restart the runtime and rerun the notebook.'
    )

TORCH_VERSION = torch.__version__.split('+')[0]
CUDA_VERSION = torch.version.cuda.replace('.', '')
CONCERTO_DIR = '/content/Concerto'

print('torch version :', torch.__version__)
print('cuda version  :', torch.version.cuda)
print('cuda wheel tag:', CUDA_VERSION)

if not os.path.exists(CONCERTO_DIR):
    !git clone https://github.com/Pointcept/Concerto.git {CONCERTO_DIR}
else:
    %cd {CONCERTO_DIR}
    !git pull
    %cd {REPO_DIR}

!pip install -q spconv-cu{CUDA_VERSION}
!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+cu{CUDA_VERSION}.html
!pip install -q huggingface_hub timm
%cd {CONCERTO_DIR}
!python setup.py install
%cd {REPO_DIR}


### 4. Symlink Shared Drive Folders

In [ ]:
!rm -f data checkpoints features pretrained
!ln -sf /content/drive/MyDrive/DL_Project/data ./data
!ln -sf /content/drive/MyDrive/DL_Project/checkpoints ./checkpoints
!ln -sf /content/drive/MyDrive/DL_Project/features ./features
!ln -sf /content/drive/MyDrive/DL_Project/pretrained ./pretrained


### 5. Verify Inputs

In [ ]:
from pathlib import Path
from src.dataset import S3DISDataset

processed_dir = Path('data/s3dis_processed')
features_dir = Path('features/s3dis_area5')
clip_emb_path = processed_dir / 'label_to_clip_embeddings.npy'
checkpoint_path = Path('pretrained/concerto_small.pth')
ckpt_dir = Path('checkpoints/mlp_s3dis')

print('processed dir exists :', processed_dir.exists(), processed_dir)
print('Area_5 exists        :', (processed_dir / 'Area_5').exists(), processed_dir / 'Area_5')
print('features dir exists  :', features_dir.exists(), features_dir)
print('CLIP embeddings file :', clip_emb_path.exists(), clip_emb_path)
print('Concerto checkpoint  :', checkpoint_path.exists(), checkpoint_path)
print('checkpoint dir exists:', ckpt_dir.exists(), ckpt_dir)

dataset = S3DISDataset(root=str(processed_dir), areas=[5])
print('Area 5 rooms:', len(dataset))
print('sample rooms:', [dataset.rooms[i].name for i in range(min(5, len(dataset.rooms)))])

assert len(dataset) > 0, 'No processed Area_5 rooms found in data/s3dis_processed.'
assert checkpoint_path.exists(), 'Missing pretrained/concerto_small.pth.'


### 6. Run the Encoder on a Real Room

This runs the real Concerto encoder on one Area 5 room before the full extraction.

In [ ]:
from pathlib import Path
import numpy as np
import torch

from src.encoder import ConcertoEncoder

sample = dataset[0]
coord = sample['coord'].astype(np.float32)
color = sample['color'].astype(np.float32)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
encoder = ConcertoEncoder(
    checkpoint_path='pretrained/concerto_small.pth',
    device=device,
)
encoder_input = {
    'coord': coord,
    'color': color,
}
with torch.no_grad():
    room_features = encoder(encoder_input)

print('room          :', sample['room'])
print('input shape   :', coord.shape)
print('feature shape :', tuple(room_features.shape))
print('all finite    :', torch.isfinite(room_features).all().item())


### 7. Extract Concerto Features for Area 5

This writes `.npz` files to `features/s3dis_area5/`. Re-run with `--overwrite` only if you want to rebuild them.

In [ ]:
!python scripts/extract_features.py --checkpoint pretrained/concerto_small.pth


In [ ]:
from pathlib import Path
import numpy as np

feature_files = sorted(Path('features/s3dis_area5').glob('*.npz'))
print('num feature files:', len(feature_files))
print('sample files     :', [path.name for path in feature_files[:5]])

assert feature_files, 'No .npz feature files were produced.'

sample = np.load(feature_files[0])
print('sample keys:', sample.files)
for key in sample.files:
    print(key, sample[key].shape, sample[key].dtype)


### 8. Generate CLIP Label Embeddings if Missing

In [ ]:
from pathlib import Path

clip_emb_path = Path('data/s3dis_processed/label_to_clip_embeddings.npy')
if clip_emb_path.exists():
    print('CLIP label embeddings already exist:', clip_emb_path)
else:
    !python src/clip_utils.py


### 9. Inspect Training Config

In [ ]:
from pathlib import Path
import yaml

config_path = Path('configs/train_mlp_s3dis.yaml')
cfg = yaml.safe_load(config_path.read_text(encoding='utf-8'))
cfg


### 10. Train the Translation Head

In [ ]:
!python src/train.py --config configs/train_mlp_s3dis.yaml


### 11. Inspect Training Outputs

In [ ]:
from pathlib import Path
import json

ckpt_dir = Path('checkpoints/mlp_s3dis')
print('checkpoint dir exists:', ckpt_dir.exists())
for path in sorted(ckpt_dir.glob('*')):
    print(path.name)

history_path = ckpt_dir / 'history.json'
if history_path.exists():
    history = json.loads(history_path.read_text(encoding='utf-8'))
    print('\nLast history rows:')
    for row in history[-5:]:
        print(row)
else:
    print('\nNo history.json found yet.')
